# Day 026 Project: AutoReporter

## What You're Building

An `AutoReporter` class that takes a dict of raw data, uses the LLM to draft a section per key, and writes both a PDF and a DOCX report.

Pipeline: `data dict → generate_sections (AI) → to_pdf + to_docx → files`

## Project Requirements

1. Implement `AutoReporter` with:
   - `generate_sections(data: dict) -> list[dict]` — one AI call per key
   - `to_pdf(title, sections, output_path)` — write PDF
   - `to_docx(title, sections, output_path)` — write DOCX
   - `generate_report(data, title, output_dir) -> dict` — full pipeline
2. Run `reporter.generate_report(SAMPLE_DATA, 'Q1 Report', '/tmp')` and store as `result`
3. Print the output file paths

**Deliverable:** PDF and DOCX report files auto-generated from raw data.

In [ ]:
import json, os
from pypdf import PdfReader
from docx import Document
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from xml.sax.saxutils import escape
import ollama

## Provided: All Helper Functions

In [ ]:
def read_pdf_pages(pdf_path: str) -> list[str]:
    reader = PdfReader(pdf_path)
    return [page.extract_text() or "" for page in reader.pages]


def read_docx_text(docx_path: str) -> str:
    doc = Document(docx_path)
    return "\n".join(p.text for p in doc.paragraphs)


def create_pdf_report(title: str, sections: list[dict], output_path: str) -> None:
    doc = SimpleDocTemplate(output_path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = [
        Paragraph(escape(title), styles["Title"]),
        Spacer(1, 0.25 * inch),
    ]
    for section in sections:
        story.append(Paragraph(escape(section["heading"]), styles["Heading1"]))
        story.append(Paragraph(escape(section["body"]), styles["Normal"]))
        story.append(Spacer(1, 0.15 * inch))
    doc.build(story)


def create_docx_report(title: str, sections: list[dict], output_path: str) -> None:
    doc = Document()
    doc.add_heading(title, level=0)
    for section in sections:
        doc.add_heading(section["heading"], level=1)
        doc.add_paragraph(section["body"])
    doc.save(output_path)


def ai_generate_section(
    topic: str,
    data_snippet: str,
    model: str = "llama3.2",
) -> dict:
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional report writer. "
                    "Generate a concise report section from the data provided. "
                    'Return JSON with exactly two keys: "heading" (a short title string) '
                    'and "body" (2-4 sentences of professional analysis). '
                    "Return only valid JSON, no explanation."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Topic: {topic}\n\n"
                    f"Data:\n{data_snippet[:500]}\n\n"
                    "Generate a report section:"
                ),
            },
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        result = json.loads(raw)
        return {
            "heading": str(result.get("heading", topic)),
            "body": str(result.get("body", "")),
        }
    except Exception:
        return {"heading": topic, "body": raw}

## Your Implementation

Implement `AutoReporter` by wiring the helper functions together.

In [ ]:
class AutoReporter:
    def __init__(self, model: str = 'llama3.2'):
        self.model = model

    def generate_sections(self, data: dict) -> list[dict]:
        # TODO: for key, value in data.items():
        #           snippet = json.dumps(value, indent=2) if isinstance(value, (dict, list)) else str(value)
        #           section = ai_generate_section(key, snippet, model=self.model)
        #           sections.append(section)
        pass

    def to_pdf(self, title: str, sections: list[dict], output_path: str) -> None:
        # TODO: create_pdf_report(title, sections, output_path)
        pass

    def to_docx(self, title: str, sections: list[dict], output_path: str) -> None:
        # TODO: create_docx_report(title, sections, output_path)
        pass

    def generate_report(self, data: dict, title: str, output_dir: str = '.') -> dict:
        # TODO: sections = self.generate_sections(data)
        # TODO: pdf_path = os.path.join(output_dir, 'report.pdf')
        # TODO: docx_path = os.path.join(output_dir, 'report.docx')
        # TODO: self.to_pdf(title, sections, pdf_path)
        # TODO: self.to_docx(title, sections, docx_path)
        # TODO: return {'pdf': pdf_path, 'docx': docx_path, 'sections': sections}
        pass

## Sample Data

In [ ]:
SAMPLE_DATA = {
    'Sales Performance': 'Q1 revenue: 1.2M, up 15 percent YoY. Top product: Widget A with 38 percent share.',
    'Market Overview':   'Total market: 50B. Our share: 2.4 percent. Three main competitors.',
}


## Run the Reporter

In [ ]:
# reporter = AutoReporter()
# result = reporter.generate_report(SAMPLE_DATA, 'Q1 Business Report', '/tmp')
# print(f"PDF:  {result['pdf']}")
# print(f"DOCX: {result['docx']}")
# for s in result['sections']:
#     print(f"  - {s['heading']}")

## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: AutoReporter defined with required methods
    try:
        assert 'AutoReporter' in globals()
        for m in ('generate_sections', 'to_pdf', 'to_docx', 'generate_report'):
            assert hasattr(AutoReporter, m), f'AutoReporter missing: {m}'
        passed += 1; print('\u2705 Check 1: AutoReporter has all required methods')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: reporter is an AutoReporter instance
    try:
        assert 'reporter' in globals()
        assert isinstance(reporter, AutoReporter)
        passed += 1; print('\u2705 Check 2: reporter is an AutoReporter')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: result is a dict with required keys
    try:
        assert 'result' in globals()
        for k in ('pdf', 'docx', 'sections'):
            assert k in result, f"result missing '{k}': {list(result)}"
        passed += 1; print('\u2705 Check 3: result has pdf/docx/sections keys')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: PDF file exists
    try:
        assert 'result' in globals()
        pdf = result.get('pdf', '')
        assert os.path.exists(pdf), f'PDF not found at {pdf}'
        size = os.path.getsize(pdf)
        assert size > 100, f'PDF too small ({size} bytes)'
        passed += 1; print(f'\u2705 Check 4: PDF exists ({size:,} bytes)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: DOCX exists and is readable
    try:
        assert 'result' in globals()
        from docx import Document as _D
        docx = result.get('docx', '')
        assert os.path.exists(docx), f'DOCX not found at {docx}'
        doc = _D(docx)
        assert len(doc.paragraphs) > 0
        passed += 1; print('\u2705 Check 5: DOCX exists and is readable')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add a `cover_page(title, subtitle, date)` helper that prepends a styled cover page to the PDF using reportlab canvas or platypus
- Add a `summarize_report(sections, model) -> str` method that asks the LLM to write a one-paragraph executive summary across all sections
- Extend `generate_sections` to accept a list instead of a dict, using each item's 'topic' key
- Add an `append_table(doc, rows: list[list[str]])` helper using `python-docx`'s `add_table` API
- Handle the case where `ai_generate_section` returns an empty body by retrying once with a more explicit prompt